# Previsão Futura de Dengue com o Melhor Modelo

Este notebook treina o melhor modelo (CatBoost) com todos os dados disponíveis e gera previsões para os próximos 12 meses.

In [ ]:
import sys
sys.path.insert(0, '../src')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from catboost import CatBoostRegressor
from skforecast.recursive import ForecasterRecursive

sns.set_theme(style='whitegrid', palette='colorblind')

## 1. Carregamento dos dados

In [ ]:
series = pd.read_csv(
    '../data/processed/dengue_monthly_sao.csv',
    index_col='date',
    parse_dates=['date'],
)['value'].asfreq('MS')

print(f'Série: {series.name}')
print(f'Período: {series.index.min().date()} a {series.index.max().date()}')

## 2. Treinamento do modelo CatBoost

In [ ]:
LAGS = 24
HORIZON = 12

forecaster = ForecasterRecursive(
    estimator=CatBoostRegressor(random_state=42, verbose=0),
    lags=LAGS,
)

forecaster.fit(y=series)
print('Modelo treinado com sucesso!')

## 3. Geração das previsões

In [ ]:
predictions = forecaster.predict(steps=HORIZON)
predictions = predictions.clip(lower=0)

print('Previsões para os próximos 12 meses:')
print(predictions.to_frame('casos_previstos').to_string())

## 4. Visualização

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

# Últimos 36 meses de dados históricos
historical = series.iloc[-36:]
ax.plot(historical.index, historical.values, label='Histórico', color='black', linewidth=1.5)
ax.plot(predictions.index, predictions.values, label='Previsão (CatBoost)', color='steelblue', linewidth=2, linestyle='--', marker='o', markersize=5)

ax.axvline(x=series.index[-1], color='gray', linestyle=':', alpha=0.7, label='Início da previsão')
ax.set_title('Previsão de Casos de Dengue — São Paulo (Próximos 12 Meses)', fontsize=13, fontweight='bold')
ax.set_ylabel('Casos mensais')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Importância das features

In [ ]:
feature_importance = pd.Series(
    forecaster.regressor_.feature_importances_,
    index=forecaster.feature_names_in_
).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
feature_importance.head(20).plot(kind='barh', color='steelblue')
plt.title('Importância das Features (CatBoost) — Top 20', fontweight='bold')
plt.xlabel('Importância')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()